### 使用langgraph构建代码助手
***
- 针对特定问题参考材料迭代式的生成代码
- 从用户指定的一组文档开始
- 使用长上下文LLM来提取它并执行RAG来回答基于它的问题
- 调用一个工具来生成结构化输出
- 将解决方案返回给用户之前，将执行两个单元测试(检查导入和代码执行)

In [1]:
! pip install -U langchain_community langchain-openai langchain-deepseek langchain langgraph bs4 langchain-anthropic

   ---------------------------------------- 0.0/574.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/574.7 kB ? eta -:--:--
   ------------------ --------------------- 262.1/574.7 kB ? eta -:--:--
   ------------------------------------ --- 524.3/574.7 kB 1.1 MB/s eta 0:00:01
   ---------------------------------------- 574.7/574.7 kB 872.7 kB/s  0:00:00
   ---------------------------------------- 0.0/911.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/911.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/911.8 kB ? eta -:--:--
   ---------------------- ----------------- 524.3/911.8 kB 1.5 MB/s eta 0:00:01
   ---------------------------------- ----- 786.4/911.8 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 911.8/911.8 kB 1.3 MB/s  0:00:00

  Attempting uninstall: langchain-protocol

    Found existing installation: langchain-protocol 0.0.18

    Uninstalling langchain-protocol-0.0.18:

      Successfully unins

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-redis 0.1.1 requires langchain-core<0.4,>=0.3, but you have langchain-core 1.7.0 which is incompatible.
langchain-redis 0.1.1 requires numpy<2,>=1, but you have numpy 2.5.2 which is incompatible.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


#### 加载文档
***

In [2]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader
import re

# 1. 定义深度清洗函数 (去除导航、页脚等噪音)
def bs4_extractor(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    
    # 移除网页中不需要的干扰标签
    for tag in soup(["script", "style", "header", "footer", "nav", "aside", "button"]):
        tag.extract()
        
    # 获取纯文本，使用换行符分隔，并去除多余空白
    return soup.get_text(separator="\n", strip=True)

# 2. 配置 Loader (使用当前有效的 LangChain 文档 URL)
url = "https://www.pinecone.io/learn/series/langchain/langchain-expression-language/?spm=a2ty_o01.29997173.0.0.44f255fbnCi92w" 

loader = RecursiveUrlLoader(
    url=url,
    max_depth=2,               # 限制深度为 2，防止抓取过多页面导致内存溢出
    extractor=bs4_extractor,   # 使用自定义的深度清洗函数
    prevent_outside=True,      # 防止抓取跳转到外部网站的链接
    check_response_status=True,# 自动忽略 404 等错误页面
)

print(f"开始递归抓取: {url} (请稍候，这可能需要几十秒)...")
docs = loader.load()

if not docs:
    print("❌ 抓取失败，未获取到任何文档。请检查网络或 URL 是否有效。")
else:
    print(f"✅ 成功抓取了 {len(docs)} 个页面！")

    # 3. 排序与拼接 (保留你原有的优秀逻辑)
    # 按 URL 排序，然后反转，确保子页面在父页面之前（这对 RAG 上下文理解有帮助）
    d_sorted = sorted(docs, key=lambda x: x.metadata["source"])
    d_reversed = list(reversed(d_sorted))
    
    concatenated_content = "\n\n\n --- \n\n\n".join(
        [doc.page_content for doc in d_reversed]
    )

    # 4. 打印结果概览
    print(f"\n📊 拼接后的总字符数: {len(concatenated_content)}")
    print("\n--- 拼接内容预览 (前 500 字) ---")
    print(concatenated_content[:500] + "...\n")

C:\Users\Lxb\AppData\Local\Temp\ipykernel_31140\940540484.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import RecursiveUrlLoader


开始递归抓取: https://www.pinecone.io/learn/series/langchain/langchain-expression-language/?spm=a2ty_o01.29997173.0.0.44f255fbnCi92w (请稍候，这可能需要几十秒)...
✅ 成功抓取了 34 个页面！

📊 拼接后的总字符数: 341768

--- 拼接内容预览 (前 500 字) ---
Terms | Pinecone
Introducing pg_pinecone: Open-Source Search for Your Postgres Data
-
Read the announcement
Legal
Website Terms of Service
May 26, 2022
The Pinecone website located at
https://www.pinecone.io
(the “
Site
”) is a copyrighted work belonging to Pinecone Systems Inc., (“
Pinecone
”, “
us
”, “
our
”, and “
we
”). The Site is for informational purposes only; it contains general information about Pinecone and is directed at businesses and their representatives seeking information on our...



In [3]:
concatenated_content

'Terms | Pinecone\nIntroducing pg_pinecone: Open-Source Search for Your Postgres Data\n-\nRead the announcement\nLegal\nWebsite Terms of Service\nMay 26, 2022\nThe Pinecone website located at\nhttps://www.pinecone.io\n(the “\nSite\n”) is a copyrighted work belonging to Pinecone Systems Inc., (“\nPinecone\n”, “\nus\n”, “\nour\n”, and “\nwe\n”). The Site is for informational purposes only; it contains general information about Pinecone and is directed at businesses and their representatives seeking information on our products and services.\nCertain features of the Site may be subject to additional guidelines, terms, or rules, which will be posted on the Site in connection with such features. All such additional terms, guidelines, and rules are incorporated by reference into these Terms of Use (“\nTerms\n”). Your access to and use of the Site is also subject to our\nPrivacy Policy\n.\nPinecone also provides a proprietary data and analytics platform, (the “\nPlatform\n”, and the provision 

设置大模型

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_deepseek import ChatDeepSeek
from pydantic import BaseModel, Field
from langchain.messages import HumanMessage
import os
from dotenv import load_dotenv
load_dotenv()

# 评分提示
code_gen_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", """
            你是一位精通LCEL(LangChain表达式语言)的编程助手。\n
            这里是LCEL文档的完整集合:\n ---------------- \n {context} \n ---------------- \n 请根据
            上述提供的文档回答用户问题。确保你提供的任何代码都可以执行，\n
            包含所有必要的导入和已定义的变量。请按照以下结构组织你的回答：首先描述代码解决方案，\n 
            然后列出导入语句，最后给出功能完整的代码块。以下是用户问题："""),
        ("placeholder", "{messages}")
    ]
)

# 数据模型
class code(BaseModel):
    """LCEL问题代码解决方案的模式"""

    prefix: str = Field(description="问题和解决方案的描述")
    imports: str = Field(description="代码块导入语句")
    code: str = Field(description="不包含导入语句的代码块")

llm = ChatDeepSeek(
    model = "deepseek-ai/DeepSeek-V4-Flash",
    temperature=0,
    api_key=os.getenv("ds_api_key"),
    base_url=os.getenv("ds_base_url")
)

code_gen_chain_oai = code_gen_prompt | llm.with_structured_output(code)
question = "如何在LCEL中构建RAG链？"
solution = code_gen_chain_oai.invoke(
    {
        "context": concatenated_content, 
        "messages": [{"role": "user", "content": question}]
        # "messages": [HumanMessage(content=question)] # 👈 使用 HumanMessage 对象
    }
)

solution

code(prefix='根据文档，LCEL（LangChain Expression Language）使用管道操作符（|）来构建链。构建RAG（Retrieval-Augmented Generation）链时，我们需要将检索器、提示模板、模型和输出解析器通过管道操作符连接起来。文档中展示了使用RunnableParallel和RunnablePassthrough来并行执行检索和传递用户问题的示例。下面是一个完整的RAG链构建方案，包含向量存储初始化、检索器设置、提示模板创建以及LCEL链的组装。', imports='from langchain.embeddings import CohereEmbeddings\nfrom langchain.vectorstores import DocArrayInMemorySearch\nfrom langchain.chat_models import ChatAnthropic\nfrom langchain.prompts import ChatPromptTemplate\nfrom langchain.schema.output_parser import StrOutputParser\nfrom langchain_core.runnables import RunnableParallel, RunnablePassthrough', code='# 设置API密钥\nCOHERE_API_KEY = "YOUR_COHERE_API_KEY"\nANTHROPIC_API_KEY = "YOUR_ANTHROPIC_API_KEY"\n\n# 1. 初始化嵌入模型\nembedding = CohereEmbeddings(\n    model="embed-english-light-v3.0",\n    cohere_api_key=COHERE_API_KEY\n)\n\n# 2. 创建向量存储（示例数据）\nvecstore = DocArrayInMemorySearch.from_texts(\n    [\n        "James\' birthday is the 7th December",\n        "James was born in 1994",\n  

In [4]:
solution.code

'# 设置API密钥\nCOHERE_API_KEY = "YOUR_COHERE_API_KEY"\nANTHROPIC_API_KEY = "YOUR_ANTHROPIC_API_KEY"\n\n# 1. 初始化嵌入模型\nembedding = CohereEmbeddings(\n    model="embed-english-light-v3.0",\n    cohere_api_key=COHERE_API_KEY\n)\n\n# 2. 创建向量存储（示例数据）\nvecstore = DocArrayInMemorySearch.from_texts(\n    [\n        "James\' birthday is the 7th December",\n        "James was born in 1994",\n        "Pinecone is a vector database for AI applications",\n        "RAG combines retrieval with generation for accurate answers"\n    ],\n    embedding=embedding\n)\n\n# 3. 创建检索器\nretriever = vecstore.as_retriever()\n\n# 4. 初始化LLM模型\nmodel = ChatAnthropic(\n    model="claude-2.1",\n    max_tokens_to_sample=512,\n    anthropic_api_key=ANTHROPIC_API_KEY\n)\n\n# 5. 创建提示模板\nprompt_str = """Answer the question below using the context:\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer: """\nprompt = ChatPromptTemplate.from_template(prompt_str)\n\n# 6. 创建输出解析器\noutput_parser = StrOutputParser()\n\n# 7. 使用Runna

In [7]:
from langchain_anthropic import ChatAnthropic
from langchain_classic.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

# Anthropic

# 强制使用工具的提示
code_gen_prompt_claude = ChatPromptTemplate.from_messages(
    [
        ("system", """
          <instructions>你是一位精通LCEL(LangChain表达式语言)的编程助手。\n
          这里是LCEL文档的完整集合:\n ---------------- \n {context} \n ---------------- \n 请根据
          上述提供的文档回答用户问题。确保你提供的任何代码都可以执行，\n
          包含所有必要的导入和已定义的变量。请按照以下结构组织你的回答：
          1) 描述代码解决方案的前言，
          2) 导入语句， 
          3) 功能完整的代码块。\n
          调用code工具来正确构建输出。<\instructions> 以下是用户问题："""),
        ("placeholder", "{messages}"),
    ]
)

# # 大语言模型
# # 注意选择使用claude 模型，因为其上下文窗口更大
# expt_llm = "claude-sonnet-4-20250514"
# claude_llm = ChatAnthropic(
#     model=expt_llm,
#     api_key=os.getenv("an_api_key"),
#     base_url=os.getenv("an_base_url"),
#     default_headers={"anthropic-beta": "tools-2026-09-02"},
# )

# structured_llm_claude = claude_llm.with_structured_output(code, include_raw=True)


llm = ChatOpenAI(
    model="Qwen/Qwen3.8-27B", 
    temperature=0,
    api_key=os.getenv("api_key"),
    base_url=os.getenv("base_url"),
)

structured_llm_claude = llm.with_structured_output(code, include_raw=True)

# 可选：检查工具使用是否出现错误
def check_claude_output(tool_output):
    """检查解析错误或未能调用工具"""

    # 解析错误
    if tool_output["parsing_error"]:
        # 报告输出和解析错误
        print("解析错误！")
        raw_output = str(tool_output["raw"].content)
        error = tool_output["parsing_error"]
        raise ValueError(
            f"解析输出时出错！请确保调用工具。输出：{raw_output}。\n 解析错误：{error}"
        )
        # 未调用工具
    elif not tool_output["parsed"]:
        print("未能调用工具！")
        raise ValueError("你没有使用提供的工具！请确保调用工具来构建输出。")
    return tool_output

# 带输出检查的链
code_chain_claude_raw = (
    code_gen_prompt_claude | structured_llm_claude | check_claude_output
)

def insert_errors(inputs):
    """在消息中插入工具解析错误"""

    # 获取错误
    error = inputs["error"]
    messages = inputs["messages"]
    messages += [
        {
            "assistant",
            f"重试。你需要修复解析错误：{error} \n\n 你必须调用提供的工具。"
        }
    ]
    return {
        "messages": messages,
        "context": inputs["context"],
    }

# 这里将作为后备链运行
fallback_chain = insert_errors | code_chain_claude_raw
N = 3 # 最大重试次数
code_gen_chain_retry = code_chain_claude_raw.with_fallbacks(
    fallbacks=[fallback_chain] * N,
    exception_key="error"
)

def parse_output(solution):
    """当我们在结构化输出中添加'include_raw=True'时，
    它将返回一个包含'raw'、'parsed'、'parsing_error'的字典。"""

    return solution["parsed"]

# 可选：带重试功能，用于纠正未能调用工具的情况
code_gen_chain = code_gen_chain_retry | parse_output

# 不重试
code_gen_chain = code_gen_prompt_claude | structured_llm_claude | parse_output


<>:20: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
<>:20: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
C:\Users\Lxb\AppData\Local\Temp\ipykernel_31140\3973225081.py:20: SyntaxWarning: "\i" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\i"? A raw string is also an option.
  调用code工具来正确构建输出。<\instructions> 以下是用户问题："""),


In [8]:
# Test
question = "在LCEL中如何构建一个RAG链？"
solution = code_gen_chain.invoke(
    {
        "context": concatenated_content,
        "messages": [{"role": "user", "content": question}]
    }
)
solution

OpenAIModelNotFoundError: Error code: 404 - {'error': {'message': 'The model `Qwen/Qwen3.8-27B` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}, 'request_id': '1929c9e6-4970-90d4-8dc3-e896e70b550c'}

定义state

In [15]:
from typing import List
from typing_extensions import TypedDict

class GraphState(TypedDict):
    """
    表示我们图的状态。

    属性:
    error: 用于控制流的二进制标志，表示是否触发了测试错误
    messages: 包含用户问题、错误消息、推理过程
    generation: 代码解决方案
    iterations: 尝试次数
    """
    
    error: str
    messages: List
    generation: str
    iterations: int


创建节点

In [16]:
from typing import List, Dict, Any
from typing_extensions import TypedDict
from langgraph.graph import END, StateGraph, START
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage

# 1. 定义图状态
class GraphState(TypedDict):
    error: str
    messages: List[BaseMessage] # 🌟 明确指定消息类型
    generation: Any             # 允许为 None 或 Pydantic 对象
    iterations: int

# 参数
max_iterations = 3
flag = "do not reflect" # "reflect" 或 "do not reflect"

# ================= 节点定义 =================

def generate(state: GraphState):
    print("---🤖 正在生成代码解决方案---")
    messages = list(state["messages"]) # 复制列表以防不可变错误
    iterations = state["iterations"]
    error = state["error"]

    if error == "yes":
        # 🌟 修复：使用 HumanMessage 对象
        messages.append(HumanMessage(content="现在，请重试。调用code工具来构建包含前言、导入和代码块的输出："))

    # 假设 code_gen_chain 和 concatenated_content 已在外部定义
    code_solution = code_gen_chain.invoke(
        {"context": concatenated_content, "messages": messages}
    )

    # 🌟 修复：使用 AIMessage 对象记录助手的回复
    if code_solution:
        messages.append(AIMessage(
            content=f"{code_solution.prefix} \n 导入：{code_solution.imports} \n 代码：{code_solution.code}"
        ))
    
    return {"generation": code_solution, "messages": messages, "iterations": iterations + 1}

def code_check(state: GraphState):
    print("---🔍 正在检查代码---")
    messages = list(state["messages"])
    code_solution = state["generation"]
    iterations = state["iterations"]

    # 🌟 防御性检查：如果大模型彻底失败，没有生成结构化代码
    if not code_solution or not hasattr(code_solution, 'imports'):
        print("---代码生成失败，未获取到有效结构---")
        messages.append(HumanMessage(content="你的输出格式完全错误，没有生成有效的代码结构。请重试。"))
        return {"generation": code_solution, "messages": messages, "iterations": iterations, "error": "yes"}

    imports = code_solution.imports
    code = code_solution.code

    # 🌟 安全修复：使用独立的字典作为 exec 的命名空间，防止污染全局或恶意代码
    safe_globals = {"__builtins__": __builtins__} 

    # 检查导入
    try:
        exec(imports, safe_globals)
    except Exception as e:
        print(f"---代码导入检查：失败 ({e})---")
        # 🌟 修复：使用 HumanMessage，绝不是 Set {"user", ...}
        messages.append(HumanMessage(content=f"你的解决方案未通过导入测试：{e}。请检查导入语句。"))
        return {"generation": code_solution, "messages": messages, "iterations": iterations, "error": "yes"}

    # 检查执行
    try:
        exec(imports + "\n" + code, safe_globals)
    except Exception as e:
        print(f"---代码块检查：失败 ({e})---")
        messages.append(HumanMessage(content=f"你的解决方案未通过代码执行测试：{e}。请修复代码逻辑。"))
        return {"generation": code_solution, "messages": messages, "iterations": iterations, "error": "yes"}

    print("---✅ 无代码测试失败，检查通过---")
    return {"generation": code_solution, "messages": messages, "iterations": iterations, "error": "no"}

def reflect(state: GraphState):
    print("---🧠 正在对错误进行反思---")
    messages = list(state["messages"])
    code_solution = state["generation"]
    iterations = state["iterations"]

    # 调用链进行反思 (注意：这里复用了 code_gen_chain，实际生产中建议用专门的反思 prompt)
    reflections = code_gen_chain.invoke(
        {"context": concatenated_content, "messages": messages}
    )
    
    if reflections:
        messages.append(AIMessage(content=f"以下是对错误的反思：{reflections.prefix}"))
        
    return {"generation": code_solution, "messages": messages, "iterations": iterations}

# ================= 边缘 (条件路由) =================

def decide_to_finish(state: GraphState):
    error = state["error"]
    iterations = state["iterations"]

    if error == "no" or iterations >= max_iterations:
        print("---🏁 决定：完成---")
        return "end"
    else:
        print("---🔄 决定：重新尝试解决方案---")
        return "reflect" if flag == "reflect" else "generate"


创建工作流

In [17]:
# ================= 构建图 =================

workflow = StateGraph(GraphState)

workflow.add_node("generate", generate)
workflow.add_node("check_code", code_check)
workflow.add_node("reflect", reflect)

workflow.add_edge(START, "generate")
workflow.add_edge("generate", "check_code")

workflow.add_conditional_edges(
    "check_code",
    decide_to_finish,
    {
        "end": END,
        "reflect": "reflect",
        "generate": "generate"
    },
)
workflow.add_edge("reflect", "generate")

app = workflow.compile()

测试

In [18]:
question = "如何并行执行两条链？"
solution = app.invoke({"messages": [("user", question)], "iterations": 0, "error": ""})

---🤖 正在生成代码解决方案---


OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'You have exceeded your current request limit. For details, see: https://help.aliyun.com/zh/model-studio/error-code#rate-limit', 'type': 'limit_requests', 'param': None, 'code': 'limit_requests'}, 'request_id': 'edf48766-32a0-9876-8f56-812a35594c27'}

In [ ]:
solution["generation"]

TypeError: 'code' object is not subscriptable

In [44]:
from typing import List, Dict, Any
from typing_extensions import TypedDict
from langgraph.graph import END, StateGraph, START
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage

# 1. 定义图状态
class GraphState(TypedDict):
    error: str
    messages: List[BaseMessage] # 🌟 明确指定消息类型
    generation: Any             # 允许为 None 或 Pydantic 对象
    iterations: int

# 参数
max_iterations = 3
flag = "do not reflect" # "reflect" 或 "do not reflect"

# ================= 节点定义 =================

def generate(state: GraphState):
    print("---🤖 正在生成代码解决方案---")
    messages = list(state["messages"]) # 复制列表以防不可变错误
    iterations = state["iterations"]
    error = state["error"]

    if error == "yes":
        # 🌟 修复：使用 HumanMessage 对象
        messages.append(HumanMessage(content="现在，请重试。调用code工具来构建包含前言、导入和代码块的输出："))

    # 假设 code_gen_chain 和 concatenated_content 已在外部定义
    code_solution = code_gen_chain.invoke(
        {"context": concatenated_content, "messages": messages}
    )

    # 🌟 修复：使用 AIMessage 对象记录助手的回复
    if code_solution:
        messages.append(AIMessage(
            content=f"{code_solution.prefix} \n 导入：{code_solution.imports} \n 代码：{code_solution.code}"
        ))
    
    return {"generation": code_solution, "messages": messages, "iterations": iterations + 1}

def code_check(state: GraphState):
    print("---🔍 正在检查代码---")
    messages = list(state["messages"])
    code_solution = state["generation"]
    iterations = state["iterations"]

    # 🌟 防御性检查：如果大模型彻底失败，没有生成结构化代码
    if not code_solution or not hasattr(code_solution, 'imports'):
        print("---代码生成失败，未获取到有效结构---")
        messages.append(HumanMessage(content="你的输出格式完全错误，没有生成有效的代码结构。请重试。"))
        return {"generation": code_solution, "messages": messages, "iterations": iterations, "error": "yes"}

    imports = code_solution.imports
    code = code_solution.code

    # 🌟 安全修复：使用独立的字典作为 exec 的命名空间，防止污染全局或恶意代码
    safe_globals = {"__builtins__": __builtins__} 

    # 检查导入
    try:
        exec(imports, safe_globals)
    except Exception as e:
        print(f"---代码导入检查：失败 ({e})---")
        # 🌟 修复：使用 HumanMessage，绝不是 Set {"user", ...}
        messages.append(HumanMessage(content=f"你的解决方案未通过导入测试：{e}。请检查导入语句。"))
        return {"generation": code_solution, "messages": messages, "iterations": iterations, "error": "yes"}

    # 检查执行
    try:
        exec(imports + "\n" + code, safe_globals)
    except Exception as e:
        print(f"---代码块检查：失败 ({e})---")
        messages.append(HumanMessage(content=f"你的解决方案未通过代码执行测试：{e}。请修复代码逻辑。"))
        return {"generation": code_solution, "messages": messages, "iterations": iterations, "error": "yes"}

    print("---✅ 无代码测试失败，检查通过---")
    return {"generation": code_solution, "messages": messages, "iterations": iterations, "error": "no"}

def reflect(state: GraphState):
    print("---🧠 正在对错误进行反思---")
    messages = list(state["messages"])
    code_solution = state["generation"]
    iterations = state["iterations"]

    # 调用链进行反思 (注意：这里复用了 code_gen_chain，实际生产中建议用专门的反思 prompt)
    reflections = code_gen_chain.invoke(
        {"context": concatenated_content, "messages": messages}
    )
    
    if reflections:
        messages.append(AIMessage(content=f"以下是对错误的反思：{reflections.prefix}"))
        
    return {"generation": code_solution, "messages": messages, "iterations": iterations}

# ================= 边缘 (条件路由) =================

def decide_to_finish(state: GraphState):
    error = state["error"]
    iterations = state["iterations"]

    if error == "no" or iterations >= max_iterations:
        print("---🏁 决定：完成---")
        return "end"
    else:
        print("---🔄 决定：重新尝试解决方案---")
        return "reflect" if flag == "reflect" else "generate"

# ================= 构建图 =================

workflow = StateGraph(GraphState)

workflow.add_node("generate", generate)
workflow.add_node("check_code", code_check)
workflow.add_node("reflect", reflect)

workflow.add_edge(START, "generate")
workflow.add_edge("generate", "check_code")

workflow.add_conditional_edges(
    "check_code",
    decide_to_finish,
    {
        "end": END,
        "reflect": "reflect",
        "generate": "generate"
    },
)
workflow.add_edge("reflect", "generate")

app = workflow.compile()

# ================= 测试运行 =================
if __name__ == "__main__":
    # 🌟 确保在运行此代码前，已经定义了 concatenated_content 和 code_gen_chain
    
    question = "如何并行执行两条链？"
    print(f"\n🚀 开始处理问题: {question}\n")
    
    # 🌟 修复：初始消息也必须使用 HumanMessage，并去掉重复的调用
    solution = app.invoke({
        "messages": [HumanMessage(content=question)], 
        "iterations": 0, 
        "error": ""
    })

    print("\n================ 最终结果 ================")
    final_gen = solution.get("generation")
    if final_gen:
        print(f"【前言】: {final_gen.prefix}")
        print(f"【导入】:\n{final_gen.imports}")
        print(f"【代码】:\n{final_gen.code}")
    else:
        print("未能生成有效的代码解决方案。")


🚀 开始处理问题: 如何并行执行两条链？

---🤖 正在生成代码解决方案---
---🔍 正在检查代码---
---代码导入检查：失败 (No module named 'langchain.chains')---
---🔄 决定：重新尝试解决方案---
---🤖 正在生成代码解决方案---
---🔍 正在检查代码---
---代码导入检查：失败 (No module named 'langchain.chains')---
---🔄 决定：重新尝试解决方案---
---🤖 正在生成代码解决方案---
---🔍 正在检查代码---
---代码导入检查：失败 (No module named 'langchain.chains')---
---🏁 决定：完成---

================ 最终结果 ================
【前言】: 在LangChain中，可以使用`RunnableParallel`来并行执行多个链。以下是一个示例，展示如何并行执行两条链：
【导入】:
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import PromptTemplate
from langchain_openai import OpenAI
from langchain.chains import LLMChain
【代码】:
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import PromptTemplate
from langchain_openai import OpenAI
from langchain.chains import LLMChain

# 初始化两个不同的提示模板
prompt1 = PromptTemplate(
    input_variables=["query"],
    template="What is the capital of {query}?"
)

prompt2 = PromptTemplate(
    input_variables=["query"],
    